# Loading Data by Activity Type
Access pattern: `participants[participant_id][activity_type][sensor_type][sensor_location]`
- Example: `participants["2995"]["Perturb"]["Trip"]["ECG"]`
- Example: `participants["2995"]["Perturb"]["Trip"]["IMU"]["Left_Thigh"]`

In [106]:
import os
import pandas as pd
import glob
from collections import defaultdict
import re

# ASSIGN COLUMN NAMES
ECG_COLS   = ["time_s", "ecg_V"]
GSS_COLS   = ["time_s", "gss_V"]
IMU_COLS   = ["time_s", "acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
LABEL_COLS = ["event", "time_s", "other"]

path = "/Users/chloechristensen/Library/Mobile Documents/com~apple~CloudDocs/School/Y5/BMEG 400K/Grand_Challenge/data"
all_files = glob.glob(os.path.join(path, "*.csv"))

# Categorize activities into ADL (Activities of Daily Living) and Perturb (Perturbation)
# ADL: Activities of Daily Living - normal daily activities
ADL_ACTIVITIES = {"Walk", "Stand", "Sit", "Lie", "Stairs", "Pick", "Jog", "JJ"}
# Perturb: Perturbations to induce fall or near fall
PERTURB_ACTIVITIES = {"Slip", "Trip", "Miss", "Hit", "Coll", "LOS", "ITD", "ITR"}

def make_activity_dict():
    return {
        "IMU": {"Back": None, "Left_Thigh": None, "Right_Thigh": None},
        "ECG": None,
        "GSS": None,
        "labels": None
    }

def make_category_dict():
    return {
        "ADL": defaultdict(make_activity_dict),
        "Perturb": defaultdict(make_activity_dict)
    }

# Structure: participants[participant_id]["ADL" or "Perturb"][activity_type][sensor_type][sensor_location]
participants = defaultdict(make_category_dict)

# Track trial identifiers for each participant/activity to keep the latest one
trial_tracker = defaultdict(lambda: defaultdict(list))  # {pid: {activity: [(trial_id, sensor, file)]}}
label_tracker = defaultdict(lambda: defaultdict(list))  # {pid: {activity: [(suffix, file)]}}

SENSOR_SUFFIXES = ["Left_Thigh", "Right_Thigh", "Back", "ECG", "GSS"]

def categorize_activity(activity):
    """Determine if activity is ADL or Perturb"""
    if activity in ADL_ACTIVITIES:
        return "ADL"
    elif activity in PERTURB_ACTIVITIES:
        return "Perturb"
    else:
        # Default to ADL for unknown activities
        return "ADL"

def parse_data_filename(base):
    """Parse data files like: data_P2995_T06_Trip_Left_Thigh.csv or data_P3033_T01A_Trip_Back.csv"""
    if not (base.startswith("data_P") and base.endswith(".csv")):
        return None
    stem = base[:-4]
    parts = stem.split("_")
    if len(parts) < 4:
        return None

    pid_part = parts[1]  # "P2995"
    if not pid_part.startswith("P") or not pid_part[1:].isdigit():
        return None
    pid = pid_part[1:]

    trial_part = parts[2]  # "T06" or "T01A" or "T01B"
    # Use regex to extract trial number and optional suffix
    match = re.match(r'T(\d+)([A-Z]?)', trial_part)
    if not match:
        return None
    trial_num = int(match.group(1))
    trial_suffix = match.group(2) if match.group(2) else ''
    trial_id = (trial_num, trial_suffix)  # e.g., (1, 'A') or (1, 'B') or (6, '')
    
    # Extract activity and sensor from remainder
    remainder = "_".join(parts[3:])
    sensor = None
    activity = None
    for sfx in SENSOR_SUFFIXES:
        tag = "_" + sfx
        if remainder.endswith(tag):
            sensor = sfx
            activity = remainder[: -len(tag)]
            break

    if sensor is None or not activity:
        return None
    return pid, trial_id, activity, sensor

def parse_label_filename(base):
    """Parse label files like: labels_P2995_Trip.csv or labels_P5690_Coll_A.csv"""
    if not (base.startswith("labels_P") and base.endswith(".csv")):
        return None
    stem = base[:-4]
    rest = stem[len("labels_P"):]  # "2995_Trip" or "5690_Coll_A"
    if "_" not in rest:
        return None
    pid_str, activity_part = rest.split("_", 1)
    if not pid_str.isdigit() or not activity_part:
        return None
    
    # Check if activity has A/B suffix (e.g., "Coll_A" or "Coll_B")
    suffix = ''
    activity = activity_part
    if activity_part.endswith('_A'):
        activity = activity_part[:-2]
        suffix = 'A'
    elif activity_part.endswith('_B'):
        activity = activity_part[:-2]
        suffix = 'B'
    
    return pid_str, activity, suffix

# First pass: collect all files and track trial identifiers
file_info = []

for f in all_files:
    base = os.path.basename(f)
    
    if os.path.getsize(f) == 0:
        continue
    
    # Label files
    lab = parse_label_filename(base)
    if lab is not None:
        pid, activity, suffix = lab
        label_tracker[pid][activity].append((suffix, f))
        continue
    
    # Data files
    parsed = parse_data_filename(base)
    if parsed is not None:
        pid, trial_id, activity, sensor = parsed
        trial_tracker[pid][activity].append((trial_id, sensor, f))
        file_info.append((pid, trial_id, activity, sensor, f))

# Second pass: load data, keeping only the highest trial identifier for each activity
for pid in trial_tracker:
    for activity in trial_tracker[pid]:
        # Find the maximum trial identifier (highest number, then highest suffix)
        # trial_id is a tuple: (trial_num, suffix) e.g., (1, 'B') > (1, 'A') > (1, '')
        max_trial_id = max(trial_id for trial_id, _, _ in trial_tracker[pid][activity])
        
        # Determine category (ADL or Perturb)
        category = categorize_activity(activity)
        
        # Load only files from the max trial
        for trial_id, sensor, f in trial_tracker[pid][activity]:
            if trial_id == max_trial_id:
                if sensor in ["Back", "Left_Thigh", "Right_Thigh"]:
                    df = pd.read_csv(f, header=None, names=IMU_COLS)
                    participants[pid][category][activity]["IMU"][sensor] = df
                elif sensor == "ECG":
                    df = pd.read_csv(f, header=None, names=ECG_COLS)
                    participants[pid][category][activity]["ECG"] = df
                elif sensor == "GSS":
                    df = pd.read_csv(f, header=None, names=GSS_COLS)
                    participants[pid][category][activity]["GSS"] = df

# Third pass: load label files (keep only the last version - B over A over no suffix)
for pid in label_tracker:
    for activity in label_tracker[pid]:
        # Sort by suffix: '' < 'A' < 'B', take the last one
        label_tracker[pid][activity].sort(key=lambda x: x[0])
        max_suffix, label_file = label_tracker[pid][activity][-1]
        
        # Determine category (ADL or Perturb)
        category = categorize_activity(activity)
        
        # Only load if the participant/activity already exists (has sensor data)
        if pid in participants and activity in participants[pid][category]:
            df = pd.read_csv(label_file, header=None, names=LABEL_COLS)
            participants[pid][category][activity]["labels"] = df

print(f"✅ Loaded data for {len(participants)} participants")
print(f"\nExample participants: {list(participants.keys())[:5]}")

# Count activities by category
adl_count = sum(len(participants[pid]["ADL"]) for pid in participants)
perturb_count = sum(len(participants[pid]["Perturb"]) for pid in participants)
print(f"\nTotal ADL activities: {adl_count}")
print(f"Total Perturb activities: {perturb_count}")

if '2995' in participants:
    print(f"\nParticipant 2995:")
    print(f"  ADL activities: {list(participants['2995']['ADL'].keys())}")
    print(f"  Perturb activities: {list(participants['2995']['Perturb'].keys())}")

✅ Loaded data for 14 participants

Example participants: ['5923', '4616', '5690', '7202', '7465']

Total ADL activities: 112
Total Perturb activities: 112

Participant 2995:
  ADL activities: ['Walk', 'Sit', 'Pick', 'Stairs', 'Lie', 'Jog', 'Stand', 'JJ']
  Perturb activities: ['Miss', 'ITD', 'Trip', 'Slip', 'LOS', 'Coll', 'ITR', 'Hit']


In [109]:
# Complete overview of all participants with ADL/Perturb breakdown
print("=" * 80)
print("COMPLETE PARTICIPANT OVERVIEW")
print("=" * 80)

for pid in sorted(participants.keys()):
    print(f"\n📊 Participant {pid}:")
    print("-" * 80)
    
    # ADL activities
    if len(participants[pid]["ADL"]) > 0:
        print(f"  ADL Activities ({len(participants[pid]['ADL'])}):")
        for activity in sorted(participants[pid]["ADL"].keys()):
            # Check if all sensors are available
            has_ecg = participants[pid]["ADL"][activity]["ECG"] is not None
            has_gss = participants[pid]["ADL"][activity]["GSS"] is not None
            has_imu = all(participants[pid]["ADL"][activity]["IMU"][s] is not None 
                         for s in ["Back", "Left_Thigh", "Right_Thigh"])
            has_labels = participants[pid]["ADL"][activity]["labels"] is not None
            
            status = "✓" if (has_ecg and has_gss and has_imu) else "⚠"
            label_status = "✓" if has_labels else "✗"
            
            print(f"    {activity:12s} [Sensors: {status}] [Labels: {label_status}]")
    
    # Perturb activities
    if len(participants[pid]["Perturb"]) > 0:
        print(f"  Perturb Activities ({len(participants[pid]['Perturb'])}):")
        for activity in sorted(participants[pid]["Perturb"].keys()):
            # Check if all sensors are available
            has_ecg = participants[pid]["Perturb"][activity]["ECG"] is not None
            has_gss = participants[pid]["Perturb"][activity]["GSS"] is not None
            has_imu = all(participants[pid]["Perturb"][activity]["IMU"][s] is not None 
                         for s in ["Back", "Left_Thigh", "Right_Thigh"])
            has_labels = participants[pid]["Perturb"][activity]["labels"] is not None
            
            status = "✓" if (has_ecg and has_gss and has_imu) else "⚠"
            label_status = "✓" if has_labels else "✗"
            
            print(f"    {activity:12s} [Sensors: {status}] [Labels: {label_status}]")

print("\n" + "=" * 80)
print(f"✅ Total: {len(participants)} participants")
print(f"   - Total ADL activities: {sum(len(participants[pid]['ADL']) for pid in participants)}")
print(f"   - Total Perturb activities: {sum(len(participants[pid]['Perturb']) for pid in participants)}")
print("=" * 80)

COMPLETE PARTICIPANT OVERVIEW

📊 Participant 1960:
--------------------------------------------------------------------------------
  ADL Activities (8):
    JJ           [Sensors: ✓] [Labels: ✗]
    Jog          [Sensors: ✓] [Labels: ✗]
    Lie          [Sensors: ✓] [Labels: ✗]
    Pick         [Sensors: ✓] [Labels: ✗]
    Sit          [Sensors: ✓] [Labels: ✗]
    Stairs       [Sensors: ✓] [Labels: ✗]
    Stand        [Sensors: ✓] [Labels: ✗]
    Walk         [Sensors: ✓] [Labels: ✗]
  Perturb Activities (8):
    Coll         [Sensors: ✓] [Labels: ✓]
    Hit          [Sensors: ✓] [Labels: ✓]
    ITD          [Sensors: ✓] [Labels: ✓]
    ITR          [Sensors: ⚠] [Labels: ✓]
    LOS          [Sensors: ✓] [Labels: ✓]
    Miss         [Sensors: ✓] [Labels: ✓]
    Slip         [Sensors: ✓] [Labels: ✓]
    Trip         [Sensors: ✓] [Labels: ✓]

📊 Participant 2070:
--------------------------------------------------------------------------------
  ADL Activities (8):
    JJ           [Sensor

# Exploring the data

Q3: Plot some examples of a fall, near-fall, and activity of daily living for each of the sensors. Make sure to note which participant and which activities the plots come from

Q4: What sources of error do you expect to encounter for each sensor in measuring falls, near-falls, and activities of daily living? Use plots to demonstrate some of these errors (again, note which participant and which activities the plots come from).

Q6: Given your understanding of the data and looking at the data itself, what information do you think will be most relevant in developing a system that can identify a fall or near-fall? (e.g. what data do you think you will rely on when developing your system and why?) Again, use some plots the justify your answer here and indicate the participant and activity. [15]

In [ ]:
# Q3
# # Plotting some of the fall, near -fall (perturbation) trials
# 2070 - has ITD and ITR 

participants_ID = participants.keys()

for i in range(len(participants_ID)):
# figure out which perturb and which ADL trials the pariticapnt has 
for activity in participants[participants_ID[i]]["Perturb"].keys():

participants["2070"]["Perturb"]["ITD"]["labels"]
participants["2070"]["Perturb"]["ITD"]["IMU"]["Back"]
participants["2070"]["Perturb"]["ITD"]["IMU"]["Left_Thigh"]
participants["2070"]["Perturb"]["ITD"]["IMU"]["Right_Thigh"]  
participants["2070"]["Perturb"]["ITD"]["ECG"]
participants["2070"]["Perturb"]["ITD"]["GSS"]  
participants["2070"]["Perturb"]["ITR"]["labels"]
participants["2070"]["Perturb"]["ITR"]["IMU"]["Back"]
participants["2070"]["Perturb"]["ITR"]["IMU"]["Left_Thigh"]
participants["2070"]["Perturb"]["ITR"]["IMU"]["Right_Thigh"]  
participants["2070"]["Perturb"]["ITR"]["ECG"]
participants["2070"]["Perturb"]["ITR"]["GSS"]

# all but jss get the 

# 2995 - has Slip and Trip
participants["2995"]["Perturb"]["Slip"]["labels"]       
participants["2995"]["Perturb"]["Slip"]["IMU"]["Back"]
participants["2995"]["Perturb"]["Slip"]["IMU"]["Left_Thigh"]
participants["2995"]["Perturb"]["Slip"]["IMU"]["Right_Thigh"]  
participants["2995"]["Perturb"]["Slip"]["ECG"]
participants["2995"]["Perturb"]["Slip"]["GSS"]



# Activity of daily living trials (ADL) trials 


In [114]:
participants_names = participants.keys()
participants_names

dict_keys(['5923', '4616', '5690', '7202', '7465', '3033', '8332', '1960', '2995', '9853', '2070', '3175', '4827', '7253'])